In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
CLEANED_DATA_PATH = PROJECT_ROOT / "data" / "cleaned"

CLEANED_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Project Root :", PROJECT_ROOT)
print("Raw Data     :", RAW_DATA_PATH)
print("Cleaned Data :", CLEANED_DATA_PATH)

Project Root : d:\cart2insights
Raw Data     : d:\cart2insights\data\raw
Cleaned Data : d:\cart2insights\data\cleaned


In [3]:
DATASETS = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv"
}

print(f"Expected datasets: {len(DATASETS)}")

Expected datasets: 9


In [4]:
data = {}

print("=" * 80)
print("LOADING RAW DATASETS")
print("=" * 80)

for table_name, filename in DATASETS.items():

    file_path = RAW_DATA_PATH / filename

    if not file_path.exists():
        print(f"❌ MISSING: {filename}")
        continue

    if file_path.stat().st_size == 0:
        print(f"❌ EMPTY: {filename}")
        continue

    try:
        df = pd.read_csv(
            file_path,
            low_memory=False
        )

        data[table_name] = df

        print(
            f"✅ {table_name:15} | "
            f"Rows: {len(df):,} | "
            f"Columns: {len(df.columns)}"
        )

    except Exception as e:
        print(f"❌ ERROR: {filename}")
        print(f"   {type(e).__name__}: {e}")

print("=" * 80)
print(f"Loaded {len(data)} / {len(DATASETS)} datasets")
print("=" * 80)

LOADING RAW DATASETS
✅ customers       | Rows: 99,441 | Columns: 5
✅ geolocation     | Rows: 1,000,163 | Columns: 5
✅ order_items     | Rows: 112,650 | Columns: 7
✅ order_payments  | Rows: 103,886 | Columns: 5
✅ order_reviews   | Rows: 99,224 | Columns: 7
✅ orders          | Rows: 99,441 | Columns: 8
✅ products        | Rows: 32,951 | Columns: 9
✅ sellers         | Rows: 3,095 | Columns: 4
✅ translation     | Rows: 71 | Columns: 2
Loaded 9 / 9 datasets


In [5]:
cleaned = {
    table_name: df.copy()
    for table_name, df in data.items()
}

print("Working copies created.")

for table_name, df in cleaned.items():
    print(f"{table_name:15} -> {df.shape}")

Working copies created.
customers       -> (99441, 5)
geolocation     -> (1000163, 5)
order_items     -> (112650, 7)
order_payments  -> (103886, 5)
order_reviews   -> (99224, 7)
orders          -> (99441, 8)
products        -> (32951, 9)
sellers         -> (3095, 4)
translation     -> (71, 2)


In [6]:
cleaning_log = []

for table_name, df in cleaned.items():

    cleaning_log.append({
        "table": table_name,
        "initial_rows": len(df),
        "initial_columns": len(df.columns)
    })

cleaning_log_df = pd.DataFrame(cleaning_log)

display(cleaning_log_df)

,table,initial_rows,initial_columns
0,customers,99441,5
1,geolocation,1000163,5
2,order_items,112650,7
3,order_payments,103886,5
4,order_reviews,99224,7
5,orders,99441,8
6,products,32951,9
7,sellers,3095,4
8,translation,71,2


In [7]:
for table_name in cleaned:

    before = len(cleaned[table_name])

    cleaned[table_name] = (
        cleaned[table_name]
        .dropna(how="all")
        .copy()
    )

    after = len(cleaned[table_name])

    print(
        f"{table_name:15} | "
        f"Removed empty rows: {before - after}"
    )

customers       | Removed empty rows: 0
geolocation     | Removed empty rows: 0
order_items     | Removed empty rows: 0
order_payments  | Removed empty rows: 0
order_reviews   | Removed empty rows: 0
orders          | Removed empty rows: 0
products        | Removed empty rows: 0
sellers         | Removed empty rows: 0
translation     | Removed empty rows: 0


In [8]:
duplicate_log = []

for table_name, df in cleaned.items():

    before = len(df)

    duplicate_count = df.duplicated().sum()

    cleaned[table_name] = (
        df
        .drop_duplicates()
        .reset_index(drop=True)
    )

    after = len(cleaned[table_name])

    duplicate_log.append({
        "table": table_name,
        "duplicates_removed": duplicate_count,
        "rows_after": after
    })

duplicate_log_df = pd.DataFrame(duplicate_log)

display(duplicate_log_df)

,table,duplicates_removed,rows_after
0,customers,0,99441
1,geolocation,261831,738332
2,order_items,0,112650
3,order_payments,0,103886
4,order_reviews,0,99224
5,orders,0,99441
6,products,0,32951
7,sellers,0,3095
8,translation,0,71


In [9]:
for table_name, df in cleaned.items():

    string_columns = df.select_dtypes(
        include=["object"]
    ).columns

    for column in string_columns:

        df[column] = df[column].apply(
            lambda x: x.strip()
            if isinstance(x, str)
            else x
        )

print("String whitespace cleaning completed.")

C:\Users\Asus-2025\AppData\Local\Temp\ipykernel_4300\1240796502.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = df.select_dtypes(
C:\Users\Asus-2025\AppData\Local\Temp\ipykernel_4300\1240796502.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-st

String whitespace cleaning completed.


C:\Users\Asus-2025\AppData\Local\Temp\ipykernel_4300\1240796502.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = df.select_dtypes(
C:\Users\Asus-2025\AppData\Local\Temp\ipykernel_4300\1240796502.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-st

In [10]:
# Orders
if "orders" in cleaned:

    cleaned["orders"]["order_status"] = (
        cleaned["orders"]["order_status"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# Payments
if "order_payments" in cleaned:

    cleaned["order_payments"]["payment_type"] = (
        cleaned["order_payments"]["payment_type"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# Customer state
if "customers" in cleaned:

    cleaned["customers"]["customer_state"] = (
        cleaned["customers"]["customer_state"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# Seller state
if "sellers" in cleaned:

    cleaned["sellers"]["seller_state"] = (
        cleaned["sellers"]["seller_state"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

print("Categorical standardization completed.")

Categorical standardization completed.


In [11]:
NUMERIC_COLUMNS = {
    "order_items": [
        "order_item_id",
        "price",
        "freight_value"
    ],

    "order_payments": [
        "payment_sequential",
        "payment_installments",
        "payment_value"
    ],

    "order_reviews": [
        "review_score"
    ],

    "products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
}

for table_name, columns in NUMERIC_COLUMNS.items():

    if table_name not in cleaned:
        continue

    for column in columns:

        if column in cleaned[table_name].columns:

            cleaned[table_name][column] = pd.to_numeric(
                cleaned[table_name][column],
                errors="coerce"
            )

print("Numeric conversion completed.")

Numeric conversion completed.


In [12]:
ORDER_DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders = cleaned["orders"]

for column in ORDER_DATE_COLUMNS:

    if column in orders.columns:

        orders[column] = pd.to_datetime(
            orders[column],
            errors="coerce"
        )

print("Order date columns converted.")

Order date columns converted.


In [13]:
reviews = cleaned["order_reviews"]

REVIEW_DATE_COLUMNS = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in REVIEW_DATE_COLUMNS:

    if column in reviews.columns:

        reviews[column] = pd.to_datetime(
            reviews[column],
            errors="coerce"
        )

print("Review date columns converted.")

Review date columns converted.


In [14]:
reviews = cleaned["order_reviews"]

invalid_review_mask = (
    ~reviews["review_score"].isin([1, 2, 3, 4, 5])
    & reviews["review_score"].notna()
)

invalid_count = invalid_review_mask.sum()

reviews.loc[
    invalid_review_mask,
    "review_score"
] = np.nan

print(
    f"Invalid review scores converted to NaN: {invalid_count}"
)

Invalid review scores converted to NaN: 0


In [15]:
INVALID_NEGATIVE_COLUMNS = {
    "order_items": [
        "price",
        "freight_value"
    ],

    "order_payments": [
        "payment_value"
    ]
}

for table_name, columns in INVALID_NEGATIVE_COLUMNS.items():

    df = cleaned[table_name]

    for column in columns:

        if column not in df.columns:
            continue

        invalid_mask = df[column] < 0

        invalid_count = invalid_mask.sum()

        df.loc[invalid_mask, column] = np.nan

        print(
            f"{table_name}.{column}: "
            f"{invalid_count} negative values handled"
        )

order_items.price: 0 negative values handled
order_items.freight_value: 0 negative values handled
order_payments.payment_value: 0 negative values handled


In [16]:
PRODUCT_DIMENSION_COLUMNS = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products = cleaned["products"]

for column in PRODUCT_DIMENSION_COLUMNS:

    if column not in products.columns:
        continue

    invalid_mask = products[column] < 0

    invalid_count = invalid_mask.sum()

    products.loc[
        invalid_mask,
        column
    ] = np.nan

    print(
        f"{column}: "
        f"{invalid_count} negative values handled"
    )

product_weight_g: 0 negative values handled
product_length_cm: 0 negative values handled
product_height_cm: 0 negative values handled
product_width_cm: 0 negative values handled


In [17]:
# Customer state
cleaned["customers"]["customer_state"] = (
    cleaned["customers"]["customer_state"]
    .fillna("UNKNOWN")
)

# Seller state
cleaned["sellers"]["seller_state"] = (
    cleaned["sellers"]["seller_state"]
    .fillna("UNKNOWN")
)

# Payment type
cleaned["order_payments"]["payment_type"] = (
    cleaned["order_payments"]["payment_type"]
    .fillna("unknown")
)

# Order status
cleaned["orders"]["order_status"] = (
    cleaned["orders"]["order_status"]
    .fillna("unknown")
)

print("Selected categorical missing values handled.")

Selected categorical missing values handled.


In [18]:
cleaned["products"]["product_category_name"] = (
    cleaned["products"]["product_category_name"]
    .fillna("unknown")
    .astype("string")
    .str.strip()
    .str.lower()
)

cleaned["translation"]["product_category_name"] = (
    cleaned["translation"]["product_category_name"]
    .astype("string")
    .str.strip()
    .str.lower()
)

cleaned["translation"]["product_category_name_english"] = (
    cleaned["translation"]["product_category_name_english"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print("Product categories standardized.")

Product categories standardized.


In [19]:
PRODUCT_RENAME_MAP = {
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
}

cleaned["products"] = (
    cleaned["products"]
    .rename(columns=PRODUCT_RENAME_MAP)
)

print("Product column names standardized.")

Product column names standardized.


In [20]:
for table_name, df in cleaned.items():

    print("\n" + "=" * 70)
    print(f"TABLE: {table_name}")
    print("=" * 70)

    display(
        df.dtypes.to_frame("dtype")
    )


TABLE: customers


,dtype
customer_id,str
customer_unique_id,str
customer_zip_code_prefix,int64
customer_city,str
customer_state,string



TABLE: geolocation


,dtype
geolocation_zip_code_prefix,int64
geolocation_lat,float64
geolocation_lng,float64
geolocation_city,str
geolocation_state,str



TABLE: order_items


...

,dtype
product_id,str
product_category_name,string
product_name_length,float64
product_description_length,float64
product_photos_qty,float64
product_weight_g,float64
product_length_cm,float64
product_height_cm,float64
product_width_cm,float64



TABLE: sellers


,dtype
seller_id,str
seller_zip_code_prefix,int64
seller_city,str
seller_state,string



TABLE: translation


,dtype
product_category_name,string
product_category_name_english,string


In [21]:
remaining_missing = []

for table_name, df in cleaned.items():

    for column in df.columns:

        missing_count = df[column].isna().sum()

        if missing_count > 0:

            remaining_missing.append({
                "table": table_name,
                "column": column,
                "missing_count": missing_count,
                "missing_percentage": round(
                    missing_count / len(df) * 100,
                    2
                )
            })

remaining_missing_df = pd.DataFrame(
    remaining_missing
)

display(
    remaining_missing_df.sort_values(
        ["table", "missing_percentage"],
        ascending=[True, False]
    )
)

,table,column,missing_count,missing_percentage
0,order_reviews,review_comment_title,87656,88.34
1,order_reviews,review_comment_message,58247,58.70
4,orders,order_delivered_customer_date,2965,2.98
3,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_approved_at,160,0.16
5,products,product_name_length,610,1.85
6,products,product_description_length,610,1.85
7,products,product_photos_qty,610,1.85
8,products,product_weight_g,2,0.01
9,products,product_length_cm,2,0.01


In [22]:
final_duplicate_check = []

for table_name, df in cleaned.items():

    final_duplicate_check.append({
        "table": table_name,
        "rows": len(df),
        "duplicate_rows": df.duplicated().sum()
    })

final_duplicate_df = pd.DataFrame(
    final_duplicate_check
)

display(final_duplicate_df)

,table,rows,duplicate_rows
0,customers,99441,0
1,geolocation,738332,0
2,order_items,112650,0
3,order_payments,103886,0
4,order_reviews,99224,0
5,orders,99441,0
6,products,32951,0
7,sellers,3095,0
8,translation,71,0


In [23]:
final_summary = []

for table_name, df in cleaned.items():

    initial_rows = (
        cleaning_log_df.loc[
            cleaning_log_df["table"] == table_name,
            "initial_rows"
        ].iloc[0]
    )

    final_summary.append({
        "table": table_name,
        "initial_rows": initial_rows,
        "final_rows": len(df),
        "rows_removed": initial_rows - len(df),
        "columns": len(df.columns),
        "missing_cells": df.isna().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

final_summary_df = pd.DataFrame(
    final_summary
)

display(final_summary_df)

,table,initial_rows,final_rows,rows_removed,columns,missing_cells,duplicate_rows
0,customers,99441,99441,0,5,0,0
1,geolocation,1000163,738332,261831,5,0,0
2,order_items,112650,112650,0,7,0,0
3,order_payments,103886,103886,0,5,0,0
4,order_reviews,99224,99224,0,7,145903,0
5,orders,99441,99441,0,8,4908,0
6,products,32951,32951,0,9,1838,0
7,sellers,3095,3095,0,4,0,0
8,translation,71,71,0,2,0,0


In [24]:
print("=" * 80)
print("SAVING CLEANED DATASETS")
print("=" * 80)

for table_name, df in cleaned.items():

    filename = DATASETS[table_name]

    output_path = CLEANED_DATA_PATH / filename

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"✅ {filename:45} "
        f"{len(df):,} rows"
    )

print("=" * 80)
print("ALL CLEANED DATASETS SAVED")
print("=" * 80)

SAVING CLEANED DATASETS
✅ olist_customers_dataset.csv                   99,441 rows
✅ olist_geolocation_dataset.csv                 738,332 rows
✅ olist_order_items_dataset.csv                 112,650 rows
✅ olist_order_payments_dataset.csv              103,886 rows
✅ olist_order_reviews_dataset.csv               99,224 rows
✅ olist_orders_dataset.csv                      99,441 rows
✅ olist_products_dataset.csv                    32,951 rows
✅ olist_sellers_dataset.csv                     3,095 rows
✅ product_category_name_translation.csv         71 rows
ALL CLEANED DATASETS SAVED


In [25]:
print("Files in data/cleaned:\n")

for file in CLEANED_DATA_PATH.iterdir():

    print(
        f"{file.name:45} "
        f"{file.stat().st_size:,} bytes"
    )

Files in data/cleaned:

olist_customers_dataset.csv                   8,661,712 bytes
olist_geolocation_dataset.csv                 44,071,918 bytes
olist_orders_dataset.csv                      16,611,357 bytes
olist_order_items_dataset.csv                 15,024,227 bytes
olist_order_payments_dataset.csv              5,737,249 bytes
olist_order_reviews_dataset.csv               14,237,954 bytes
olist_products_dataset.csv                    2,833,152 bytes
olist_sellers_dataset.csv                     166,686 bytes
product_category_name_translation.csv         2,612 bytes
